# Fine-Tuning A Toy Instruction Task

Fine-tuning changes the training distribution from generic next-token text to a narrower prompt-response format. This notebook keeps the setup deliberately small so the mechanics of supervised fine-tuning stay inspectable.


## Problem: What Must This Component Compute?

In supervised fine-tuning, the model still predicts the next token, but the token stream now represents formatted instruction-response examples rather than generic pretraining text. The input tensor still has shape `(B, T)`, and the model still emits logits of shape `(B, T, V)`.

We will first write the operation directly with tensors so every dimension is visible. Only after the numerical checks pass will we wrap the operation in a reusable module.

Why this matters later: when an architecture paper claims to improve alignment, instruction following, or data efficiency, it is usually changing one of these constraints: how examples are formatted, which tokens contribute to loss, how much data diversity is available, or how decoding is evaluated after tuning.


## 1. What Supervised Fine-Tuning Changes

The model architecture does not change. What changes is the dataset: prompts and desired responses are serialized into a format that teaches the model to continue with an answer after an instruction prefix. This is still next-token prediction, but on a more targeted distribution.


In [ ]:
from llm_from_scratch.data import toy_instruction_examples

for prompt, response in toy_instruction_examples():
    print(f"### Instruction:\n{prompt}\n\n### Response:\n{response}\n")


## 2. Prompt-Response Formatting

A minimal format makes the task boundary explicit. We will use a short textual template with `### Instruction:` and `### Response:` tags so the continuation target is visible in the raw character stream.


In [ ]:
import torch

from llm_from_scratch.configs import ModelConfig, TrainConfig, get_device, set_seed
from llm_from_scratch.data import split_tokens, toy_instruction_examples
from llm_from_scratch.generate import generate
from llm_from_scratch.model import TinyGPT
from llm_from_scratch.tokenization import CharTokenizer
from llm_from_scratch.train import train_steps

def format_example(prompt: str, response: str) -> str:
    return f"### Instruction:\n{prompt}\n\n### Response:\n{response}\n"

formatted = [format_example(prompt, response) for prompt, response in toy_instruction_examples()]
tokenizer = CharTokenizer.from_text(''.join(formatted))
runtime_device = get_device()
set_seed(123)
tokens = torch.tensor(tokenizer.encode(''.join(formatted * 4)), dtype=torch.long)
train_tokens, val_tokens = split_tokens(tokens, train_fraction=0.85)

assert len(train_tokens) > 96
assert len(val_tokens) > 32

{
    "runtime_device": runtime_device.type,
    "num_examples": len(formatted),
    "vocab_size": tokenizer.vocab_size,
    "train_len": len(train_tokens),
    "val_len": len(val_tokens),
}


## 3. Why This Toy Task Is Not Instruction Tuning At Production Scale

Real instruction tuning uses far more examples, more careful curation, multiple task types, stronger validation, and often response-only loss masking. This notebook is intentionally tiny: it is for understanding the mechanism, not for producing a competitive aligned assistant.


## 4. Loss Masking Discussion

In many supervised fine-tuning pipelines, prompt tokens are present in the context but excluded from the loss so the model is optimized mainly on response tokens. Our package training loop currently computes loss on every target token, so we will illustrate masking with a tensor-only demo and then keep the actual fine-tuning run simple.


In [ ]:
def prompt_response_mask(text: str) -> torch.Tensor:
    response_prefix = "### Response:\n"
    split_idx = text.index(response_prefix) + len(response_prefix)
    mask = torch.zeros(len(text), dtype=torch.long)
    mask[split_idx:] = 1
    return mask

single_mask = prompt_response_mask(formatted[0])
response_prefix = "### Response:\n"
split_idx = formatted[0].index(response_prefix) + len(response_prefix)

assert single_mask[:split_idx].sum().item() == 0
assert torch.all(single_mask[split_idx:] == 1)

{
    "prompt_chars": split_idx,
    "response_chars": int(single_mask.sum().item()),
    "mask_preview": single_mask[:24].tolist(),
}


## 5. Small Fine-Tuning Loop Using Toy Examples

We can now run a tiny fine-tuning loop on repeated formatted examples. The goal is not quality; it is to verify that the existing package APIs can train on instruction-shaped text and then generate from an instruction prefix.


In [ ]:
set_seed(123)
cfg = ModelConfig(vocab_size=tokenizer.vocab_size, block_size=96, n_embd=48, n_head=4, n_layer=2, dropout=0.0)
model = TinyGPT(cfg)
history = train_steps(
    model,
    train_tokens,
    val_tokens,
    TrainConfig(
        batch_size=8,
        max_steps=20,
        eval_interval=10,
        eval_batches=2,
        learning_rate=1e-2,
        weight_decay=0.0,
        grad_clip=1.0,
        seed=123,
    ),
    runtime_device,
)

finite_history = [all(torch.isfinite(torch.tensor([row['train'], row['val']])).tolist()) for row in history]
assert len(history) == 2
assert all(finite_history)

{
    "runtime_device": runtime_device.type,
    "history": history,
}


In [ ]:
set_seed(123)
prompt = format_example('Define attention in one sentence.', '')
prompt_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=runtime_device)
generated_ids = generate(model, prompt_ids, max_new_tokens=24, temperature=0.8, top_k=8)
decoded = tokenizer.decode(generated_ids[0].detach().cpu().tolist())

assert generated_ids.shape[1] == prompt_ids.shape[1] + 24

{
    "runtime_device": runtime_device.type,
    "prompt_length": prompt_ids.shape[1],
    "generated_text": decoded,
}


## Abstraction Bridge

The fine-tuning path above reuses the same `TinyGPT`, `train_steps`, and `generate` helpers as pretraining-style experiments. The main difference is the data contract: instruction formatting, optional masking policy, and evaluation questions are all different even though the optimizer and model class stay the same.


## Exercise Checkpoint 1

1. Why can supervised fine-tuning be implemented with the same next-token loss used in pretraining?
2. What information does the prompt-response template add that a plain concatenated string would hide?


## Exercise Checkpoint 2

1. Why might you want prompt tokens in the context but excluded from the loss?
2. What failure mode would you expect if the instruction and response examples were formatted inconsistently across the dataset?
